## Laboratorium 7


### Zbiór danych

Zbiór danych znajduje się w katalogu `dataset/*`. Jest to zmodyfikowany zbiór danych znajdujący się pod adresem: <https://archive.ics.uci.edu/ml/datasets/leaf>.

### Przesyłanie zadań

Wszystkie pliki należy spakować archiwizatorem **zip** i przesłać za pośrednictwem platformy WIKAMP. Poniżej oczekiwana zawartość archiwum:

```
+-- 📂 [IMIE.NAZWISKO].zip
    +-- 📜 Lab[xx].ipynb
    +-- 📂 dataset
        +-- 📜 dataset.npz
        +-- 📜 ReadMe.pdf
```

# Zadanie

1. Wybierz dane dotyczące 10 pierwszych gatunków liści (identyfikatory: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10).

2. Znormalizuj dane, przekształcając wartości cech do zakresu [0, 1].

3. Wybierz $n$ najlepszych cech (przy użyciu metod poznanych na poprzednim laboratorium np.  `SelectKBest`).

4. Przeprowadź walidację krzyżową z wykorzystaniem sprawdzianu *k*-krotnego z losowaniem warstwowym (*stratified sampling*):  
   - Skorzystaj z klasy [`StratifiedKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html).
   - Znajdź liczbę podziałów (*folds*) w zakresie od 2 do 10, dla której otrzymany wynik walidacji (np. accuracy) jest najlepszy.
   - Wykorzystaj klasyfikator *k*-najbliższych sąsiadów (kNN):  
   - Znajdź optymalną liczbę sąsiadów (*k*), dla której dokładność klasyfikacji (accuracy) jest najwyższa.
  
5. Najlepszy uzyskany model (wg. `accuracy`) przetestuj na zbiorze testowym.

6. Zapisz na dysku:
    - najlepszy wytrenowany model
    - zestaw danych wykorzystanych do wytrenowania tego modelu
  
7. Wczytaj dane z pliku i wytrenuj model ponownie, następnie przetestuj na zbiorze testowym. Wyniki powinny być takie same w pkt. 5.




In [101]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier

In [32]:
with open('.\dataset\dataset.npz', 'rb') as f:
    data = np.load(f)
    train, test = data['train'], data['test']
train.shape, test.shape

((2244, 16), (1496, 16))

In [35]:
train_df = pd.DataFrame(train)
test_df = pd.DataFrame(test)

In [38]:
train_df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,36.0,1.0,0.390930,1.102500,0.733510,0.720220,0.694740,0.179540,0.076072,1.053200,0.059213,0.157470,0.024197,0.009541,0.000247,1.204200
1,5.0,4.0,0.936710,2.415100,0.729800,0.817930,0.864910,0.334390,0.080539,1.180500,0.048722,0.120510,0.014314,0.003998,0.000372,1.308300
2,29.0,1.0,0.837500,1.951200,0.490500,0.968000,0.982460,0.651380,0.016224,0.047908,0.005119,0.035621,0.001267,0.000322,0.000011,0.235140
3,35.0,3.0,0.907550,2.582000,0.623940,0.968370,0.998250,0.556740,0.031714,0.183050,0.079387,0.162130,0.025613,0.007412,0.000699,1.695100
4,32.0,7.0,0.884850,2.239800,0.557540,0.979970,0.998250,0.679740,0.009129,0.015166,0.025658,0.087206,0.007548,0.002152,0.000179,0.751540
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2239,31.0,3.0,0.985425,12.367959,0.965230,0.802650,0.975840,0.043345,0.196795,7.280538,0.013304,0.035287,0.002212,0.000395,0.000086,0.316025
2240,29.0,6.0,0.732868,2.193675,0.451017,0.895804,0.917298,0.656747,0.030173,0.147090,0.012712,0.048700,0.000179,0.000484,-0.000019,0.363004
2241,12.0,8.0,0.972539,2.580435,0.604563,0.943933,0.927550,0.412375,0.027216,0.261505,0.093553,0.169002,0.032028,0.010133,0.000466,2.262180
2242,28.0,11.0,0.869988,2.412292,0.474942,0.894748,1.136941,0.723794,0.003889,0.034966,0.113536,0.209715,0.040135,0.010886,0.001081,2.261769


In [59]:
# first 10 classes
train_df_first_10 = train_df[train_df[0].isin([1,2,3,4,5,6,7,8,9,10])]
test_df_first_10 = test_df[test_df[0].isin([1,2,3,4,5,6,7,8,9,10])]
train_df_first_10.shape, test_df_first_10.shape

((671, 16), (517, 16))

In [63]:
X_train, y_train = train_df_first_10.drop(columns=[0]), train_df_first_10[0]
X_test, y_test = test_df_first_10.drop(columns=[0]), test_df_first_10[0]

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((671, 15), (671,), (517, 15), (517,))

In [64]:
sorted(y_train.unique()), sorted(y_test.unique())

([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0],
 [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0])

In [65]:
#normalize data
scaler = MinMaxScaler()
X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)

In [69]:
X_train_norm = pd.DataFrame(X_train_norm)
X_train_norm.min(), X_train_norm.max()

(0     0.0
 1     0.0
 2     0.0
 3     0.0
 4     0.0
 5     0.0
 6     0.0
 7     0.0
 8     0.0
 9     0.0
 10    0.0
 11    0.0
 12    0.0
 13    0.0
 14    0.0
 dtype: float64,
 0     1.0
 1     1.0
 2     1.0
 3     1.0
 4     1.0
 5     1.0
 6     1.0
 7     1.0
 8     1.0
 9     1.0
 10    1.0
 11    1.0
 12    1.0
 13    1.0
 14    1.0
 dtype: float64)

In [79]:
# select k best
n = 5
selector = SelectKBest(k=n).fit(X_train_norm, y_train)
X_train_k = selector.transform(X_train_norm) 
X_test_k = selector.transform(X_test_norm)

In [89]:
selector.get_feature_names_out()

array(['x2', 'x3', 'x6', 'x7', 'x8'], dtype=object)

In [91]:
X_train_k.shape, X_test_k.shape

((671, 5), (517, 5))

In [105]:
# Stratified K-Fold cross-validator.

results = []

for k_fold in range(2,10+1):
        
    for n_neighbors in range(1,20):
        
        fold_acc = []
        skf = StratifiedKFold(n_splits=k_fold)
        
        for train_index, test_index in skf.split(X_train_k, y_train):
            x_train_fold, x_test_fold = X_train_k[train_index], X_train_k[test_index]
            y_train_fold, y_test_fold = y_train.iloc[train_index], y_train.iloc[test_index]
        
            neigh = KNeighborsClassifier(n_neighbors=n_neighbors)
            neigh.fit(x_train_fold, y_train_fold)
            y_pred = neigh.predict(x_test_fold)
            acc = accuracy_score(y_test_fold, y_pred)
            fold_acc.append(acc)
            
        # average accuracy across all folds
        avg_acc = np.mean(fold_acc)
            
        results.append({
            'KFold': k_fold, 
            'KNeigh': n_neighbors, 
            'Accuracy': avg_acc
        })

In [ ]:
results_df = pd.DataFrame(results)
results_df

,KFold,KNeigh,Accuracy
0,2,1,0.780952
1,2,2,0.757085
2,2,3,0.795864
3,2,4,0.794336
4,2,5,0.807756
...,...,...,...
166,10,15,0.824056
167,10,16,0.821071
168,10,17,0.810645
169,10,18,0.818130


In [125]:
best_result = max(results_df.iterrows(), key=lambda x: x[1]["Accuracy"])
# best_result = max(results_df.itertuples(), key=lambda x: x.Accuracy)
best_result

(107,
 KFold        7.000000
 KNeigh      13.000000
 Accuracy     0.834555
 Name: 107, dtype: float64)